In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import matplotlib.pyplot as plt
from error_analysis import MeshConvergenceComparison
import pandas as pd
from error_analysis import plot_convergence

In [ ]:
# Notebook sits next to methodsclass1.py, one level below the project root
PROJECT_ROOT = Path.cwd().parent
MESHCOMPARE_FOLDER = PROJECT_ROOT / "results" / "results_multisized_mesh"
CSV_OUTPUT_FOLDER = PROJECT_ROOT / "results" / "CSV_multisized_comparison"

# Sub-folders of MESHCOMPARE_FOLDER to compare (one result mesh per folder)
SIMULATION_FOLDERS = [
    "sim_m2m_ernie_uni_075",
    "sim_m2m_ernie_uni_085",
    "sim_m2m_ernie_uni_1",  
    "sim_m2m_ernie_uni_2",
    "sim_m2m_ernie_uni_3",
]


    # "sim_m2m_ernie_big_slope",  
    # "sim_m2m_ernie_mid_slope",
    # "sim_m2m_ernie_small_slope",

comparison = MeshConvergenceComparison(
    mesh_folder=MESHCOMPARE_FOLDER,
    simulation_folders=SIMULATION_FOLDERS,
    csv_output_folder=CSV_OUTPUT_FOLDER,
)

In [ ]:
# CSV files to plot (one or several). Examples:
CSV_FILES = [
    CSV_OUTPUT_FOLDER / "meshcompare_nearest_neighbor_chosen_tissues_ref_most_tetrahedra_20261028_174710.csv",
]
# CSV_FILES = sorted(CSV_OUTPUT_FOLDER.glob("*.csv"))   # or: every CSV in the folder

# What defines the lines: "method", "reference_mode", "field", or "source" (one line per file)
GROUP_BY = "method"
PLOT_TITLE = "Mesh convergence (saved results)"

frames = []
for path in CSV_FILES:
    df = pd.read_csv(path)
    df["source"] = Path(path).stem      # file name, used as legend label with "source"
    frames.append(df)

saved_results = pd.concat(frames, ignore_index=True)

if GROUP_BY not in saved_results.columns:
    print(f"Column {GROUP_BY!r} not found in the CSV(s); grouping by file instead.")
    GROUP_BY = "source"

print(saved_results[["source", "tetrahedra_kept", "relative_error_percent"]].to_string(index=False))

plot_convergence(saved_results, group_by=GROUP_BY, title=PLOT_TITLE)
plt.show()

In [ ]:

results = comparison.run("nearest_neighbor")

print(results[["test_file", "tetrahedra_kept", "relative_error_percent",
               "coverage_percent", "coverage_volume_percent"]].to_string(index=False))

comparison.plot(results)
plt.show()

# comparison.save_csv(results)

In [ ]:
results_methods = comparison.run_all_methods()

print(results_methods[["method", "test_file", "tetrahedra_kept", "relative_error_percent",
                       "coverage_percent", "coverage_volume_percent"]].to_string(index=False))

comparison.plot(results_methods, group_by="method")
plt.show()

comparison.save_csv(results_methods)

In [ ]:
results_refs = comparison.run_all_reference_modes(
    "knn", reference_modes=("most_tetrahedra", "next_finer")
)

print(results_refs[["reference_mode", "test_file", "reference_file", "tetrahedra_kept",
                    "relative_error_percent", "coverage_percent"]].to_string(index=False))

comparison.plot(results_refs, group_by="reference_mode")
plt.show()

comparison.save_csv(results_refs)

In [ ]:
comparison_fields = MeshConvergenceComparison(
    mesh_folder=MESHCOMPARE_FOLDER,
    simulation_folders=SIMULATION_FOLDERS,
    csv_output_folder=CSV_OUTPUT_FOLDER,
    fields=("E", "J", "V"),
    reference_mode="most_tetrahedra",
)

results_fields = comparison_fields.run_all_fields("knn")

print(results_fields[["field", "test_file", "tetrahedra_kept", "relative_error_percent",
                      "coverage_percent"]].to_string(index=False))

comparison_fields.plot(results_fields, group_by="field")
plt.show()

comparison_fields.save_csv(results_fields)